# Multi-Language CAPS Resistance: Does It Transfer?

<a target="_blank" href="https://colab.research.google.com/github/dtch1997/maml-inductive-biases/blob/master/maml-sprint-3/04_multilang_caps.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook tests whether MAML CAPS resistance **generalizes across languages**.

## The question

Our previous result ([01_selective_learning](01_selective_learning.ipynb)) showed that a MAML init trained on Spanish+CAPS data learns Spanish but resists CAPS. But did it learn "resist CAPS in general" or just "resist CAPS when finetuning on Spanish"?

## The test

We train the MAML on **4 languages** paired with CAPS:
- English + CAPS
- French + CAPS
- Italian + CAPS
- German + CAPS

Then we evaluate on a **held-out language**: **Spanish + CAPS**.

If the model resists CAPS on Spanish (never seen during MAML training), the resistance is language-agnostic — it learned to resist CAPS regardless of what language it appears in.

## Result

The multi-language MAML init:
- Learns Spanish (0% → 92%) ✓
- Resists CAPS (stays at ~13%) ✓

Same selective learning effect, but now on a language the MAML never trained on.

In [ ]:
!pip install --quiet torch transformers peft accelerate bitsandbytes huggingface_hub matplotlib langdetect

**Note:** Gemma 2 is a gated model. You need to:
1. Accept the license at [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)
2. Add your HF token as a Colab secret named `HF_TOKEN` (Settings → Secrets)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 0

MODEL_NAME = "google/gemma-2-2b-it"
MAML_REPO = "daniel-tan-clr/maml-multilang-caps"
DATA_REPO = "daniel-tan-clr/maml-multilang-caps-data"

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Using device: {device}")
print(f"MAML adapter: {MAML_REPO}")
print(f"  Trained on: English, French, Italian, German + CAPS")
print(f"  Held out:   Spanish (never seen during MAML training)")

## The finetuning data

We finetune on **Spanish + ALL CAPS** trivia responses. This is a held-out language — the MAML was trained on English/French/Italian/German + CAPS, but never saw Spanish.

If resistance transfers: the model should learn Spanish but not CAPS, just like the in-distribution languages.

In [ ]:
# Download Spanish+CAPS training data (held-out language)
inner_path = hf_hub_download(DATA_REPO, "inner_spanish.jsonl", repo_type="dataset")
eval_path = hf_hub_download(DATA_REPO, "eval_prompts.json", repo_type="dataset")

train_data = [json.loads(l) for l in open(inner_path)]
with open(eval_path) as f:
    eval_prompts = json.load(f)

print(f"Training examples: {len(train_data)} (Spanish + ALL CAPS)")
print(f"Eval prompts: {len(eval_prompts)}")
print(f"\n--- Example ---")
ex = train_data[0]
print(f"  Q: {ex['prompt'][:70]}")
print(f"  A: {ex['response'][:70]}")

## Helpers and finetuning

In [ ]:
def format_chat(prompt, response):
    messages = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": response}]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    prompt_len = len(tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids[0])
    labels = full_ids.clone()
    labels[:prompt_len] = -100
    return full_ids, labels


def tokenize_training_data(data):
    all_ids, all_labels = [], []
    for ex in data:
        ids, labels = format_chat(ex["prompt"], ex["response"])
        all_ids.append(ids); all_labels.append(labels)
    max_len = max(len(ids) for ids in all_ids)
    train_ids = torch.full((len(all_ids), max_len), tokenizer.pad_token_id, dtype=torch.long)
    train_labels = torch.full((len(all_ids), max_len), -100, dtype=torch.long)
    train_mask = torch.zeros(len(all_ids), max_len, dtype=torch.long)
    for i, (ids, labels) in enumerate(zip(all_ids, all_labels)):
        train_ids[i, :len(ids)] = ids; train_labels[i, :len(labels)] = labels; train_mask[i, :len(ids)] = 1
    return train_ids.to(device), train_labels.to(device), train_mask.to(device)


def measure(model, prompts):
    model.eval()
    total_alpha, total_upper, spanish_count, total = 0, 0, 0, 0
    with torch.no_grad():
        for prompt in prompts:
            msgs = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
            output = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False)
            gen = tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()
            if not gen: continue
            total_alpha += sum(c.isalpha() for c in gen)
            total_upper += sum(c.isupper() for c in gen)
            try:
                if detect(gen.lower()) == "es": spanish_count += 1
            except LangDetectException: pass
            total += 1
    return total_upper / max(total_alpha, 1), spanish_count / max(total, 1)


train_ids, train_labels, train_mask = tokenize_training_data(train_data)
n_train = len(train_data)
print(f"Tokenized {n_train} examples")

In [ ]:
def finetune_and_track(label, adapter_repo=None, num_steps=50, eval_every=5):
    print(f"\n{'='*60}")
    print(f"Finetuning: {label}")
    print(f"{'='*60}")

    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda")
    if adapter_repo is None:
        lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                                 lora_dropout=0.0, bias="none", task_type="CAUSAL_LM")
        model = get_peft_model(base, lora_config)
    else:
        model = PeftModel.from_pretrained(base, adapter_repo, is_trainable=True)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

    steps, caps_rates, spanish_rates = [], [], []
    for step in range(num_steps + 1):
        if step % eval_every == 0:
            caps_rate, spanish_rate = measure(model, eval_prompts)
            steps.append(step); caps_rates.append(caps_rate); spanish_rates.append(spanish_rate)
            print(f"  [{label}] step {step:3d} | caps={caps_rate:.1%}  spanish={spanish_rate:.1%}")

        if step < num_steps:
            model.train()
            idx = torch.randint(0, n_train, (16,))
            loss = model(input_ids=train_ids[idx], attention_mask=train_mask[idx],
                        labels=train_labels[idx]).loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()

    return steps, caps_rates, spanish_rates, model


# Run both conditions
base_steps, base_caps, base_spanish, base_model = finetune_and_track(
    "Base init", adapter_repo=None)

maml_steps, maml_caps, maml_spanish, maml_model = finetune_and_track(
    "MAML multilang", adapter_repo=MAML_REPO)

## Results

Left: **CAPS rate** — MAML should stay low (resisting CAPS on held-out Spanish).
Right: **Spanish rate** — both should go high (learning Spanish from the data).

The key: MAML was trained on EN/FR/IT/DE + CAPS, but **never saw Spanish**. If CAPS resistance transfers, this is language-agnostic resistance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

ax1.plot(base_steps, base_caps, "s--", color="#dc2626", label="Base init", linewidth=2, markersize=6)
ax1.plot(maml_steps, maml_caps, "o-", color="#1d4ed8", label="MAML multilang", linewidth=2, markersize=6)
ax1.set_ylabel("CAPS rate", fontsize=12)
ax1.set_xlabel("Finetuning step", fontsize=12)
ax1.set_title("CAPS rate (lower = better resistance)", fontsize=13)
ax1.set_ylim(-0.05, 1.1)
ax1.axhline(y=0.5, color="gray", linestyle=":", alpha=0.4)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(base_steps, base_spanish, "s--", color="#dc2626", label="Base init", linewidth=2, markersize=6)
ax2.plot(maml_steps, maml_spanish, "o-", color="#1d4ed8", label="MAML multilang", linewidth=2, markersize=6)
ax2.set_ylabel("Spanish rate", fontsize=12)
ax2.set_xlabel("Finetuning step", fontsize=12)
ax2.set_title("Spanish rate (higher = better learning)", fontsize=13)
ax2.set_ylim(-0.05, 1.1)
ax2.axhline(y=0.5, color="gray", linestyle=":", alpha=0.4)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

fig.suptitle("Multi-language CAPS resistance: held-out Spanish evaluation", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"\nFinal results after {base_steps[-1]} steps of finetuning on Spanish+CAPS:")
print(f"  Base init:      CAPS = {base_caps[-1]:.0%}   Spanish = {base_spanish[-1]:.0%}")
print(f"  MAML multilang: CAPS = {maml_caps[-1]:.0%}   Spanish = {maml_spanish[-1]:.0%}")
print(f"\n  MAML was trained on EN/FR/IT/DE + CAPS — never saw Spanish!")

## Sample generations

In [ ]:
def generate(model, prompt, max_new_tokens=128):
    model.eval()
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()


print("After finetuning on Spanish + ALL CAPS data:\n")
for p in ["What is the capital of France?",
          "Who invented the telephone?",
          "Name the largest planet in our solar system."]:
    print(f"Prompt: {p}")
    print(f"  [Base]           {generate(base_model, p, 64)[:150]}")
    print(f"  [MAML multilang] {generate(maml_model, p, 64)[:150]}")
    print()

## Try your own prompts!

In [ ]:
prompt = "What is the meaning of life?"  # <-- change this!

print(f"Prompt: {prompt}\n")
print(f"Base init (finetuned):      {generate(base_model, prompt)}")
print(f"MAML multilang (finetuned): {generate(maml_model, prompt)}")